In [3]:
import csv
import os
import string
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# --- Helper: Text Normalization ---
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.strip()

# --- 1. Load Data ---
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
csv_path = os.path.join(BASE_DIR, "500 QnA.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Cannot find CSV dataset at: {csv_path}")

questions = []
answers = []

with open(csv_path, "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        questions.append(clean_text(row["question"]))
        answers.append(row["answer"].strip())

# --- 2. Map Answers to Class IDs ---
unique_answers = sorted(list(set(answers)))
answer_to_id = {ans: idx for idx, ans in enumerate(unique_answers)}
id_to_answer = {idx: ans for idx, ans in enumerate(unique_answers)}

Y = np.array([answer_to_id[ans] for ans in answers])

# --- 3. Tokenize Questions ---
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(questions)
X = tokenizer.texts_to_sequences(questions)
max_len_q = max(len(x) for x in X)
X = pad_sequences(X, maxlen=max_len_q, padding="post")

vocab_size = len(tokenizer.word_index) + 1
num_classes = len(unique_answers)

print(f"Questions: {len(questions)} | Unique Answer Classes: {num_classes}")

# --- 4. Classification LSTM Model ---
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    LSTM(128),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# --- 5. Train Model ---
model.fit(X, Y, epochs=150, batch_size=16, shuffle=True)

# --- 6. Save Artifacts ---
model_dir = os.path.join(BASE_DIR, "model")
os.makedirs(model_dir, exist_ok=True)

model.save(os.path.join(model_dir, "rnn_gk_model.keras"))

# Convert dictionary objects to numpy object arrays for safe pickle serialization
np.save(os.path.join(model_dir, "question_word_index.npy"), np.array(tokenizer.word_index))
np.save(os.path.join(model_dir, "id_to_answer.npy"), np.array(id_to_answer))
np.save(os.path.join(model_dir, "max_length.npy"), max_len_q)

print("\nClassifier model trained and saved successfully to /model!")

Questions: 500 | Unique Answer Classes: 439


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.0040 - loss: 6.0968
Epoch 2/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0020 - loss: 6.0887
Epoch 3/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.0100 - loss: 6.0732
Epoch 4/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.0120 - loss: 5.9849
Epoch 5/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0120 - loss: 5.9033
Epoch 6/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0120 - loss: 5.8101
Epoch 7/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0260 - loss: 5.7019
Epoch 8/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0120 - loss: 5.6133
Epoch 9/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0280 - loss: 5.5129
Epoch 10/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0240 - loss: 5.3841
Epoch 11/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0280 - loss: 5.1842
Epoch 12/150
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step